In [ ]:
from datetime import datetime
from getpass import getpass
import random

admin_rdm_url = 'admin.staging.rdm.example.com'

idp_name_integrated_admin = None
idp_username_integrated_admin = None
idp_password_integrated_admin = None

email_search = None
default_result_path = None
close_on_fail = False
transition_timeout = 60000

In [ ]:
if idp_username_integrated_admin is None:
    idp_username_integrated_admin = input(prompt=f'Username for {idp_name_integrated_admin}')
if idp_password_integrated_admin is None:
    idp_password_integrated_admin = getpass(prompt=f'Password for {idp_username_integrated_admin}@{idp_name_integrated_admin}')
(len(idp_username_integrated_admin), len(idp_password_integrated_admin))

In [ ]:
import tempfile

work_dir = tempfile.mkdtemp()
if default_result_path is None:
    default_result_path = work_dir
work_dir

# GDPR_Delete

- サブシステム名: GDPRDeleteによるユーザ削除
- ページ/アドオン: 管理者
- 機能分類: ユーザ削除
- シナリオ名: GDPRDeleteによるユーザ削除
- 用意するテストデータ: URL一覧、アカウント (統合管理者)

In [ ]:
import importlib
import pandas as pd

import scripts.playwright
importlib.reload(scripts.playwright)

from scripts.playwright import *
from scripts import grdm

await init_pw_context(close_on_fail=close_on_fail, last_path=default_result_path)

## ウェブブラウザの新規プライベートウィンドウでGakunin RDM管理者のトップページを表示する

管理者トップページが表示されること

In [ ]:
async def _step(page):
    await page.goto(admin_rdm_url)

    await expect(page.locator('.login-logo')).to_be_visible(timeout=30000)

await run_pw(_step)

## ログイン情報を用いてGakuNin RDMにログインする

個人の管理者ページが表示されること

In [ ]:
async def _step(page):
    await scripts.grdm.login_as_admin(
        page, idp_name_integrated_admin, idp_username_integrated_admin, idp_password_integrated_admin, transition_timeout=transition_timeout
    )

    await expect(page.locator('//*[contains(@class, "btn-danger") and contains(text(), "ログアウト")]')).to_be_enabled(timeout=transition_timeout)

await run_pw(_step)

## 「ユーザ管理」から「ユーザ管理」選択し、 ページを表示する

ユーザ検索画面が表示される

In [ ]:
async def _step(page):
    await page.locator('//a[@href = "#collapseUsers"]').click()
    await page.locator('//a[@href = "/users/"]').click()

    await expect(page.locator('//input[@name = "guid"]').first).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## 「ユーザ検索」画面のeメール欄へメールと入力し、「検索」ボタンを押下する

該当のユーザーが表示される

In [ ]:
async def _step(page):
    await page.locator('//input[@name = "email"]').fill(email_search)
    await page.locator('//input[@type = "submit"]').click()

    await expect(page.locator(f'//td[contains(text(), "{email_search}")]').first).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## 「ユーザ詳細」画面で「GDPRアカウントの削除」ボタンをクリックする

「このユーザーをGDPR削除してもよろしいですか？」ダイアログが表示されること

In [ ]:
async def _step(page):
    await page.get_by_role("link", name="GDPRアカウントの削除").click()
    await expect( page.locator("#deleteModal h3")).to_contain_text("このユーザーをGDPR削除してもよろしいですか？")

await run_pw(_step)

## 「確認」ボタンをクリックする

「User <uid> was successfully GDPR deleted」メッセージが表示されること

In [ ]:
async def _step(page):
    uid = page.url.rstrip("/").split("/")[-1]
    print(uid)

    await page.locator('#deleteModal input[type="submit"][value="確認"]').click()
    await expect(page.get_by_role("alert").first).to_contain_text(f"User {uid} was successfully GDPR deleted")

await run_pw(_step)

## 「ユーザ管理」から「ユーザ管理」選択し、 ページを表示する

ユーザ検索画面が表示される

In [ ]:
async def _step(page):
    await page.locator('//a[@href = "#collapseUsers"]').click()
    await page.get_by_role("link", name="ユーザ管理").click()

    await expect(page.locator('//input[@name = "guid"]').first).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## 「ユーザ検索」画面のeメール欄へメールと入力し、「検索」ボタンを押下する

「User with email address <email> not found.」が表示されること

In [ ]:
async def _step(page):
    await page.locator('//input[@name = "email"]').fill(email_search)
    await page.locator('//input[@type = "submit"]').click()

    await expect(page.get_by_text(f'User with email address {email_search} not found.')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

終了処理を実施。

In [ ]:
await finish_pw_context()

In [ ]:
!rm -fr {work_dir}